# تمرین ۲ — طراحی و پیاده‌سازی ترنسفورمر از صفر برای طبقه‌بندی متن (AG News)

**Transformer Encoder from Scratch** — همه‌ی اجزا فقط با `nn.Linear`، `nn.LayerNorm`، `nn.Embedding` و عملیات ماتریسی تنسورها ساخته شده‌اند.
هیچ‌کدام از `torch.nn.TransformerEncoder`، `torch.nn.MultiheadAttention`، `F.scaled_dot_product_attention` یا مدل‌های آماده‌ی Hugging Face استفاده **نشده‌اند**.

| بخش | محتوا |
|---|---|
| ۰ | تنظیمات و فراپارامترها |
| ۱ | داده، توکنایزر و واژگان |
| ۲ | کلاس‌های مجزای مدل (Embedding, Positional Encoding, Attention, Multi-Head, FFN, Encoder Layer, Encoder) |
| ۳ | سر طبقه‌بندی ([CLS] یا Mean Pooling) + Cross-Entropy |
| ۴ | تست ابعاد تنسورها در لایه‌ها |
| ۵ | آموزش و منحنی‌های همگرایی |
| ۶ | ارزیابی روی تست و ماتریس درهم‌ریختگی |
| ۷ | مصورسازی وزن‌های توجه (Heatmap) |

## ۰. تنظیمات، کتابخانه‌ها و فراپارامترها

In [ ]:
import os, re, json, math, time, random, csv, urllib.request
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("torch", torch.__version__, "| device:", device, "| threads:", torch.get_num_threads())

FIG_DIR, RES_DIR, DATA_DIR = "figures", "results", "data"
for d in (FIG_DIR, RES_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
CONFIG = dict(
    d_model=128,        # بعد امبدینگ و مدل
    num_heads=4,        # تعداد سرهای توجه  ->  d_k = d_v = d_model / h = 32
    d_ff=256,           # بعد لایه‌ی میانی FFN
    num_layers=2,       # تعداد لایه‌های کدگذار N
    dropout=0.1,
    max_len=128,        # حداکثر طول توکن‌ها (با احتساب [CLS])
    vocab_size=30000,   # حداکثر اندازه‌ی واژگان
    min_freq=2,
    pooling="cls",      # "cls" یا "mean"
    batch_size=64,
    epochs=6,
    lr=5e-4,
    weight_decay=0.01,
    warmup_ratio=0.06,
    grad_clip=1.0,
    val_size=6000,      # بخشی از داده‌ی آموزش برای اعتبارسنجی
)
CONFIG["d_k"] = CONFIG["d_v"] = CONFIG["d_model"] // CONFIG["num_heads"]
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
for k, v in CONFIG.items():
    print(f"{k:>13}: {v}")

## ۱. داده: AG News

- ۱۲۰٬۰۰۰ نمونه‌ی آموزش و ۷٬۶۰۰ نمونه‌ی تست، ۴ کلاس متوازن.
- متن ورودی = عنوان + توضیح خبر.
- ابتدا از کتابخانه‌ی `datasets` (فقط برای **داده**، نه مدل) تلاش می‌شود؛ اگر در دسترس نبود، فایل‌های CSV استاندارد (Zhang et al., 2015) دانلود می‌شوند.

In [ ]:
AG_CSV_URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/{}.csv"

def _read_csv(path):
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.reader(f):
            label, title, desc = int(row[0]) - 1, row[1], row[2]     # برچسب‌ها 1..4 -> 0..3
            texts.append(f"{title} . {desc}")
            labels.append(label)
    return texts, labels

def load_ag_news():
    try:
        from datasets import load_dataset
        ds = load_dataset("fancyzhx/ag_news")
        return ((list(ds["train"]["text"]), list(ds["train"]["label"])),
                (list(ds["test"]["text"]), list(ds["test"]["label"])))
    except Exception as e:
        print("datasets در دسترس نیست، استفاده از CSV:", type(e).__name__)
    out = []
    for split in ("train", "test"):
        path = os.path.join(DATA_DIR, f"{split}.csv")
        if not os.path.exists(path):
            urllib.request.urlretrieve(AG_CSV_URL.format(split), path)
        out.append(_read_csv(path))
    return tuple(out)

(train_texts, train_labels), (test_texts, test_labels) = load_ag_news()
print(len(train_texts), "train |", len(test_texts), "test")
print("class counts (train):", Counter(train_labels))
print("example:", CLASS_NAMES[train_labels[0]], "|", train_texts[0][:150])

### توکنایزر و واژگان

توکنایزر ساده‌ی سطح کلمه (حروف کوچک + جدا کردن علائم) و واژگان از روی داده‌ی **آموزش** ساخته می‌شود.
توکن‌های ویژه: `[PAD]=0`، `[UNK]=1`، `[CLS]=2`. توکن `[CLS]` به ابتدای هر جمله اضافه می‌شود.

In [ ]:
class SimpleTokenizer:
    # کلمات/اعداد (با آپستروف داخلی) یا یک علامت نگارشی
    _pattern = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[^\sa-z0-9]")

    @staticmethod
    def clean(text):
        text = text.lower()
        text = re.sub(r"&lt;.*?&gt;|<[^>]+>", " ", text)          # تگ‌های HTML
        text = re.sub(r"#36;", "$", text).replace("\\", " ")
        text = re.sub(r"&\w+;|#\d+;", " ", text)
        return text

    def __call__(self, text):
        return self._pattern.findall(self.clean(text))


class Vocab:
    PAD, UNK, CLS = "[PAD]", "[UNK]", "[CLS]"

    def __init__(self, token_lists, max_size, min_freq):
        counter = Counter(tok for toks in token_lists for tok in toks)
        specials = [self.PAD, self.UNK, self.CLS]
        words = [w for w, c in counter.most_common(max_size - len(specials)) if c >= min_freq]
        self.itos = specials + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id, self.unk_id, self.cls_id = 0, 1, 2

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [self.stoi.get(t, self.unk_id) for t in tokens]

    def decode(self, ids):
        return [self.itos[i] for i in ids]


tokenizer = SimpleTokenizer()
train_tokens = [tokenizer(t) for t in train_texts]
test_tokens = [tokenizer(t) for t in test_texts]
vocab = Vocab(train_tokens, CONFIG["vocab_size"], CONFIG["min_freq"])

lengths = np.array([len(t) for t in train_tokens])
print("vocab size:", len(vocab))
print(f"tokens/sample: mean={lengths.mean():.1f}, median={np.median(lengths):.0f}, "
      f"p95={np.percentile(lengths, 95):.0f}, max={lengths.max()}")
print(f"truncated by max_len={CONFIG['max_len']}: {(lengths + 1 > CONFIG['max_len']).mean()*100:.2f}% of samples")
print(train_tokens[0])

In [ ]:
class AGNewsDataset(Dataset):
    def __init__(self, token_lists, labels, vocab, max_len):
        # [CLS] در ابتدای هر جمله؛ بریدن طول به max_len
        self.ids = [[vocab.cls_id] + vocab.encode(toks)[: max_len - 1] for toks in token_lists]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.ids[i], self.labels[i]


def collate_batch(batch, pad_id=0):
    # padding پویا تا طول بلندترین جمله‌ی batch
    seqs, labels = zip(*batch)
    T = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), T), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, : len(s)] = torch.tensor(s, dtype=torch.long)
    pad_mask = input_ids != pad_id            # True = توکن واقعی ، False = پدینگ
    return input_ids, pad_mask, torch.tensor(labels, dtype=torch.long)


# جدا کردن مجموعه‌ی اعتبارسنجی از آموزش
perm = np.random.RandomState(SEED).permutation(len(train_tokens))
val_idx, tr_idx = perm[: CONFIG["val_size"]], perm[CONFIG["val_size"]:]

train_ds = AGNewsDataset([train_tokens[i] for i in tr_idx], [train_labels[i] for i in tr_idx], vocab, CONFIG["max_len"])
val_ds = AGNewsDataset([train_tokens[i] for i in val_idx], [train_labels[i] for i in val_idx], vocab, CONFIG["max_len"])
test_ds = AGNewsDataset(test_tokens, test_labels, vocab, CONFIG["max_len"])

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, collate_fn=collate_batch)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}  steps/epoch={len(train_loader)}")

## ۲. بخش اول — کلاس‌های مجزای اجزای مدل

| کلاس | ورودی → خروجی | توضیح |
|---|---|---|
| `TokenEmbedding` | `(B,T)` → `(B,T,d)` | جدول امبدینگ × $\sqrt{d_{model}}$ |
| `PositionalEncoding` | `(B,T,d)` → `(B,T,d)` | کدگذاری سینوسی موقعیت |
| `ScaledDotProductAttention` | `Q,K,V:(B,h,T,d_k)` → `(B,h,T,d_v)` + `(B,h,T,T)` | $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$ با ماسک پدینگ |
| `MultiHeadAttention` | `(B,T,d)` → `(B,T,d)` | ۴ نگاشت خطی $W_Q,W_K,W_V,W_O$ + تقسیم به سرها |
| `PositionwiseFeedForward` | `(B,T,d)` → `(B,T,d)` | $\mathrm{Linear}(d\to d_{ff})\to\mathrm{GELU}\to\mathrm{Linear}(d_{ff}\to d)$ |
| `EncoderLayer` | `(B,T,d)` → `(B,T,d)` | MHA + FFN با اتصال باقیمانده و LayerNorm |
| `TransformerEncoder` | `(B,T)` → `(B,T,d)` | Embedding + PE + N لایه‌ی کدگذار |

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.scale = math.sqrt(d_model)

    def forward(self, input_ids):                       # (B, T)
        return self.emb(input_ids) * self.scale         # (B, T, d_model)


class PositionalEncoding(nn.Module):
    # PE(pos, 2i) = sin(pos / 10000^(2i/d)),  PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        pos = torch.arange(max_len, dtype=torch.float).unsqueeze(1)                       # (L, 1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))     # (1, L, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):                               # (B, T, d_model)
        return self.dropout(x + self.pe[:, : x.size(1)])

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, key_mask=None):
        # q, k: (B, h, T, d_k)   v: (B, h, T, d_v)   key_mask: (B, T)  True = توکن واقعی
        d_k = q.size(-1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)          # (B, h, T, T)
        if key_mask is not None:
            # هیچ query‌ای نباید به کلیدهای پدینگ توجه کند
            scores = scores.masked_fill(~key_mask[:, None, None, :], float("-inf"))
        attn = torch.softmax(scores, dim=-1)                                     # (B, h, T, T)
        out = torch.matmul(self.dropout(attn), v)                                # (B, h, T, d_v)
        return out, attn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model باید بر num_heads بخش‌پذیر باشد"
        self.h, self.d_k = num_heads, d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention(dropout)
        self.attn_weights = None                       # برای مصورسازی نگه داشته می‌شود

    def _split_heads(self, x):                         # (B, T, d_model) -> (B, h, T, d_k)
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.d_k).transpose(1, 2)

    def _merge_heads(self, x):                         # (B, h, T, d_k) -> (B, T, d_model)
        B, _, T, _ = x.shape
        return x.transpose(1, 2).contiguous().view(B, T, self.h * self.d_k)

    def forward(self, x, key_mask=None):
        q = self._split_heads(self.W_q(x))
        k = self._split_heads(self.W_k(x))
        v = self._split_heads(self.W_v(x))
        out, attn = self.attention(q, k, v, key_mask)
        self.attn_weights = attn.detach()
        return self.W_o(self._merge_heads(out)), attn

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):                              # (B, T, d_model)
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))


class EncoderLayer(nn.Module):
    # Post-LN مانند مقاله‌ی اصلی:  x = LN(x + Dropout(MHA(x)));  x = LN(x + Dropout(FFN(x)))
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, key_mask=None):
        attn_out, attn = self.self_attn(x, key_mask)
        x = self.norm1(x + self.drop1(attn_out))
        x = self.norm2(x + self.drop2(self.ffn(x)))
        return x, attn


class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len, dropout=0.1, pad_id=0):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, d_model, pad_id)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

    def forward(self, input_ids, key_mask=None):
        x = self.pos_encoding(self.embedding(input_ids))
        attentions = []
        for layer in self.layers:
            x, attn = layer(x, key_mask)
            attentions.append(attn)
        return x, attentions                           # (B, T, d_model), N × (B, h, T, T)

## ۳. بخش دوم — سر طبقه‌بندی (Classification Head)

هر دو روش پیاده شده و با `CONFIG["pooling"]` انتخاب می‌شود:
1. **`cls`**: بردار خروجی توکن `[CLS]` (موقعیت ۰) در لایه‌ی آخر.
2. **`mean`**: میانگین خروجی همه‌ی توکن‌های **غیرپدینگ**: $\; h = \frac{\sum_t m_t\, x_t}{\sum_t m_t}$

سپس یک لایه‌ی خطی نهایی $d_{model}\to 4$ و تابع زیان **Cross-Entropy**.

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, d_model=128, num_heads=4, d_ff=256,
                 num_layers=2, max_len=128, dropout=0.1, pooling="cls", pad_id=0):
        super().__init__()
        assert pooling in ("cls", "mean")
        self.pooling = pooling
        self.encoder = TransformerEncoder(vocab_size, d_model, num_heads, d_ff, num_layers, max_len, dropout, pad_id)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def pool(self, h, key_mask):
        if self.pooling == "cls":
            return h[:, 0]                                             # (B, d_model)
        m = key_mask.unsqueeze(-1).type_as(h)                          # (B, T, 1)
        return (h * m).sum(1) / m.sum(1).clamp(min=1.0)                # (B, d_model)

    def forward(self, input_ids, key_mask=None, return_attn=False):
        h, attentions = self.encoder(input_ids, key_mask)
        logits = self.classifier(self.dropout(self.pool(h, key_mask)))  # (B, num_classes)
        return (logits, attentions) if return_attn else logits


def build_model():
    return TransformerClassifier(
        vocab_size=len(vocab), num_classes=len(CLASS_NAMES), d_model=CONFIG["d_model"],
        num_heads=CONFIG["num_heads"], d_ff=CONFIG["d_ff"], num_layers=CONFIG["num_layers"],
        max_len=CONFIG["max_len"], dropout=CONFIG["dropout"], pooling=CONFIG["pooling"], pad_id=vocab.pad_id,
    ).to(device)

model = build_model()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_emb = model.encoder.embedding.emb.weight.numel()
print(model)
print(f"\ntrainable params: {n_params:,}  (embedding: {n_emb:,} | transformer+head: {n_params - n_emb:,})")

# بررسی اینکه از ماژول‌های ممنوعه استفاده نشده است
banned = (nn.MultiheadAttention, nn.TransformerEncoder, nn.TransformerEncoderLayer, nn.Transformer)
assert not any(isinstance(m, banned) for m in model.modules())
print("✓ no torch.nn.MultiheadAttention / TransformerEncoder used")

## ۴. تست ابعاد تنسورها در لایه‌ها

یک batch واقعی از داده از تک‌تک اجزا عبور داده می‌شود و ابعاد چاپ و با `assert` بررسی می‌شود.
همچنین بررسی می‌شود که (۱) جمع هر سطر ماتریس توجه = ۱ و (۲) وزن توجه روی توکن‌های پدینگ = ۰ باشد.

In [ ]:
model.eval()
input_ids, key_mask, labels = next(iter(train_loader))
input_ids, key_mask = input_ids[:4].to(device), key_mask[:4].to(device)
B, T = input_ids.shape
d, h, dk = CONFIG["d_model"], CONFIG["num_heads"], CONFIG["d_k"]

def show(name, t, expected):
    assert tuple(t.shape) == tuple(expected), f"{name}: {tuple(t.shape)} != {expected}"
    print(f"{name:<42} {str(tuple(t.shape)):<20} ✓")

with torch.no_grad():
    enc = model.encoder
    show("input_ids", input_ids, (B, T))
    show("key_mask (padding mask)", key_mask, (B, T))
    e = enc.embedding(input_ids);                     show("TokenEmbedding", e, (B, T, d))
    x = enc.pos_encoding(e);                          show("+ PositionalEncoding", x, (B, T, d))

    mha = enc.layers[0].self_attn
    q = mha._split_heads(mha.W_q(x));                 show("Q after W_q + split heads", q, (B, h, T, dk))
    k = mha._split_heads(mha.W_k(x));                 show("K after W_k + split heads", k, (B, h, T, dk))
    v = mha._split_heads(mha.W_v(x));                 show("V after W_v + split heads", v, (B, h, T, dk))
    a_out, a = mha.attention(q, k, v, key_mask);      show("ScaledDotProductAttention output", a_out, (B, h, T, dk))
    show("attention weights", a, (B, h, T, T))
    show("merge heads", mha._merge_heads(a_out), (B, T, d))
    m_out, _ = mha(x, key_mask);                      show("MultiHeadAttention (after W_o)", m_out, (B, T, d))
    show("PositionwiseFeedForward", enc.layers[0].ffn(x), (B, T, d))
    show("fc1 inside FFN (d_ff)", enc.layers[0].ffn.fc1(x), (B, T, CONFIG["d_ff"]))

    y = x
    for i, layer in enumerate(enc.layers):
        y, _ = layer(y, key_mask);                    show(f"EncoderLayer {i + 1}", y, (B, T, d))
    H, atts = enc(input_ids, key_mask);               show("TransformerEncoder output", H, (B, T, d))
    show("pooled sentence vector", model.pool(H, key_mask), (B, d))
    logits = model(input_ids, key_mask);              show("logits", logits, (B, len(CLASS_NAMES)))

    # بررسی‌های صحت
    assert torch.allclose(a.sum(-1), torch.ones(B, h, T, device=device), atol=1e-5)
    pad_attn = a.masked_select(~key_mask[:, None, None, :].expand_as(a))
    assert pad_attn.numel() == 0 or pad_attn.abs().max() == 0
    print("\n✓ each attention row sums to 1   ✓ zero attention on [PAD] keys")
    print("padding tokens in this batch:", int((~key_mask).sum()))
model.train();

## ۵. آموزش

- بهینه‌ساز **AdamW** با warmup خطی و سپس کاهش کسینوسی نرخ یادگیری، gradient clipping = 1.0.
- تابع زیان: `nn.CrossEntropyLoss`.
- در هر اپوک Loss و Accuracy روی آموزش و اعتبارسنجی ثبت و بهترین مدل (بر اساس دقت اعتبارسنجی) ذخیره می‌شود.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
total_steps = CONFIG["epochs"] * len(train_loader)
warmup_steps = int(CONFIG["warmup_ratio"] * total_steps)

def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def run_epoch(model, loader, train=True):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for input_ids, key_mask, labels in loader:
            input_ids, key_mask, labels = input_ids.to(device), key_mask.to(device), labels.to(device)
            logits = model(input_ids, key_mask)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                optimizer.step()
                scheduler.step()
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(-1) == labels).sum().item()
            n += labels.size(0)
    return total_loss / n, correct / n

In [ ]:
history = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc", "epoch_time")}
best_val, ckpt_path = 0.0, os.path.join(RES_DIR, "best_model.pt")

for epoch in range(1, CONFIG["epochs"] + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, train=True)
    va_loss, va_acc = run_epoch(model, val_loader, train=False)
    dt = time.time() - t0
    for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc, dt)):
        history[k].append(v)
    flag = ""
    if va_acc > best_val:
        best_val = va_acc
        torch.save(model.state_dict(), ckpt_path)
        flag = "  ← best"
    print(f"epoch {epoch}/{CONFIG['epochs']} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} | {dt:.0f}s{flag}")

print(f"\nbest val acc: {best_val:.4f}  | total training time: {sum(history['epoch_time'])/60:.1f} min")

In [ ]:
epochs = range(1, CONFIG["epochs"] + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["val_loss"], "s-", label="validation")
axes[0].set(title="Cross-Entropy Loss", xlabel="epoch", ylabel="loss")
axes[1].plot(epochs, history["train_acc"], "o-", label="train")
axes[1].plot(epochs, history["val_acc"], "s-", label="validation")
axes[1].set(title="Accuracy", xlabel="epoch", ylabel="accuracy")
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(); ax.set_xticks(list(epochs))
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## ۶. ارزیابی روی داده‌ی تست و ماتریس درهم‌ریختگی

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for input_ids, key_mask, labels in test_loader:
        logits = model(input_ids.to(device), key_mask.to(device))
        all_preds.append(logits.argmax(-1).cpu()); all_labels.append(labels)
y_pred, y_true = torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()
test_loss, test_acc = run_epoch(model, test_loader, train=False)

print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}\n")
report_txt = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(report_txt)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm / cm.sum(1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set(title="Confusion matrix (counts)", xlabel="Predicted", ylabel="True")
sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set(title="Confusion matrix (row-normalized)", xlabel="Predicted", ylabel="True")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"most frequent confusion: true={CLASS_NAMES[i]} → predicted={CLASS_NAMES[j]} ({off[i, j]} samples)")

## ۷. بخش چهارم — مصورسازی وزن‌های توجه

دو جمله از مجموعه‌ی **تست** انتخاب می‌شود: یک خبر ورزشی و یک خبر علمی/فناوری (هر دو کوتاه تا heatmap خوانا باشد و هر دو درست طبقه‌بندی شده باشند).
برای هر جمله ماتریس توجه $T\times T$ هر ۴ سر در **لایه‌ی آخر** رسم می‌شود. چون طبقه‌بندی از روی بردار `[CLS]` انجام می‌شود، سطر `[CLS]` (اینکه `[CLS]` به کدام کلمات نگاه می‌کند) جداگانه هم نمایش داده می‌شود.

In [ ]:
def encode_text(text):
    ids = [vocab.cls_id] + vocab.encode(tokenizer(text))[: CONFIG["max_len"] - 1]
    input_ids = torch.tensor([ids], device=device)
    return input_ids, torch.ones_like(input_ids, dtype=torch.bool)

@torch.no_grad()
def predict_with_attention(text):
    input_ids, key_mask = encode_text(text)
    logits, attentions = model(input_ids, key_mask, return_attn=True)
    probs = torch.softmax(logits, -1)[0].cpu().numpy()
    tokens = vocab.decode(input_ids[0].tolist())
    return tokens, probs, [a[0].cpu().numpy() for a in attentions]   # N × (h, T, T)

def find_by_title(title):
    return next(i for i, t in enumerate(test_texts) if t.startswith(title))

def pick_example(label, keywords, min_len=15, max_len=30):
    # جایگزین: اولین جمله‌ی کوتاه تست از کلاس داده‌شده که کلمه‌ی کلیدی دارد و درست پیش‌بینی شده
    for idx, (toks, y) in enumerate(zip(test_tokens, test_labels)):
        if y == label and min_len <= len(toks) <= max_len and any(kw in toks for kw in keywords) and y_pred[idx] == y:
            return idx
    raise ValueError("no example found")

# جملات انتخابی: یک خبر ورزشی با نام تیم (Yankees) و یک خبر فناوری با واژه‌های تخصصی (Skype, software, Wi-Fi)
sports_idx = find_by_title("Yankees' Brown Has Successful Surgery")
tech_idx = find_by_title("Skype releases Pocket PC software")
if y_pred[sports_idx] != test_labels[sports_idx]:
    sports_idx = pick_example(1, {"yankees", "sox", "nba", "league", "coach", "cup", "olympic"})
if y_pred[tech_idx] != test_labels[tech_idx]:
    tech_idx = pick_example(3, {"microsoft", "software", "linux", "google", "nasa", "intel", "internet"})
examples = [("sports", sports_idx), ("scitech", tech_idx)]
for name, idx in examples:
    print(f"[{name}] test #{idx} ({CLASS_NAMES[test_labels[idx]]}): {test_texts[idx]}")

In [ ]:
def plot_heads(text, name, layer=-1):
    tokens, probs, atts = predict_with_attention(text)
    A = atts[layer]                                       # (h, T, T)
    h = A.shape[0]
    T = len(tokens)
    fig, axes = plt.subplots(2, 2, figsize=(17, 17), gridspec_kw={"hspace": 0.35})
    for head, ax in enumerate(axes.flat[:h]):
        sns.heatmap(A[head], xticklabels=tokens, yticklabels=tokens, cmap="viridis", square=True,
                    cbar_kws={"shrink": 0.6}, ax=ax, vmin=0)
        ax.set_title(f"Layer {len(atts) if layer == -1 else layer + 1} – Head {head + 1}", fontsize=13)
        ax.set_xlabel("Key (attended token)"); ax.set_ylabel("Query")
        ax.tick_params(axis="x", rotation=90, labelsize=9); ax.tick_params(axis="y", rotation=0, labelsize=9)
    pred = CLASS_NAMES[int(probs.argmax())]
    fig.suptitle(f"Last-layer attention ({name}) — predicted: {pred} (p={probs.max():.3f})", fontsize=15)
    fig.subplots_adjust(top=0.95)
    plt.savefig(os.path.join(FIG_DIR, f"attention_heads_{name}.png"), dpi=130, bbox_inches="tight")
    plt.show()
    return tokens, probs, atts

def plot_cls_row(tokens, atts, name):
    A = atts[-1]                                          # (h, T, T)
    rows = np.vstack([A[:, 0, :], A[:, 0, :].mean(0, keepdims=True)])   # سطر [CLS] هر سر + میانگین
    fig, ax = plt.subplots(figsize=(max(10, 0.55 * len(tokens)), 4.6))
    sns.heatmap(rows, xticklabels=tokens, yticklabels=[f"head {i + 1}" for i in range(A.shape[0])] + ["mean"],
                cmap="magma_r", annot=len(tokens) <= 26, fmt=".2f", annot_kws={"size": 7}, ax=ax)
    ax.tick_params(axis="y", rotation=0)
    ax.set_title(f"Attention from [CLS] to every token — last layer ({name})")
    ax.tick_params(axis="x", rotation=90)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"attention_cls_{name}.png"), dpi=150, bbox_inches="tight")
    plt.show()

def top_tokens(tokens, atts, k=6):
    w = atts[-1][:, 0, :].mean(0)                         # میانگین توجه [CLS] روی سرها
    order = [i for i in np.argsort(-w) if i != 0][:k]
    return [(tokens[i], float(w[i])) for i in order]

attention_summary = {}
for name, idx in examples:
    text = test_texts[idx]
    tokens, probs, atts = plot_heads(text, name)
    plot_cls_row(tokens, atts, name)
    per_head = {f"head{hh + 1}": [(tokens[i], float(atts[-1][hh, 0, i]))
                                  for i in np.argsort(-atts[-1][hh, 0, :])[:4] if i != 0][:3]
                for hh in range(atts[-1].shape[0])}
    attention_summary[name] = dict(index=int(idx), text=text, tokens=tokens,
                                   probs={c: float(p) for c, p in zip(CLASS_NAMES, probs)},
                                   top_cls_tokens=top_tokens(tokens, atts), top_per_head=per_head)
    print(f"\n[{name}] probs:", {c: round(float(p), 3) for c, p in zip(CLASS_NAMES, probs)})
    print("top tokens attended by [CLS] (mean over heads):",
          ", ".join(f"{t} ({w:.3f})" for t, w in attention_summary[name]["top_cls_tokens"]))
    for hname, lst in per_head.items():
        print(f"   {hname}: " + ", ".join(f"{t} ({w:.2f})" for t, w in lst))

In [ ]:
# ذخیره‌ی نتایج برای گزارش
results = dict(
    config=CONFIG, vocab_size=len(vocab), n_params=n_params, n_params_no_emb=n_params - n_emb,
    history=history, best_val_acc=best_val, test_loss=test_loss, test_acc=test_acc,
    classification_report=report_dict, confusion_matrix=cm.tolist(), attention=attention_summary,
    device=str(device), torch_version=torch.__version__,
)
with open(os.path.join(RES_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("saved:", sorted(os.listdir(FIG_DIR)), "+ results/metrics.json")

## ۸. تحلیل

تحلیل کامل نتایج (جدول فراپارامترها، منحنی‌ها، ماتریس درهم‌ریختگی و تفسیر heatmap‌ها) در فایل `report/report.pdf` آمده است.